In [1]:
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode
from qdrant_client.http.models import SearchParams
from langchain_openai import OpenAIEmbeddings
from datetime import datetime, timezone
from getpass import getpass
from pathlib import Path
import pandas as pd
import json
import os

In [2]:
COLLECTION_NAME = "20260424_big_pdfs_removed"
DENSE_MODEL_NAME = "BAAI/bge-m3"
SPARSE_MODEL_NAME = "Qdrant/bm25"
K = 10
AVG_LEN_SPARSE_FR = 255.62018086625417 # TODO : avg_len a recuperer autrement

if not os.getenv("BGE_API_KEY_EMBEDDINGS"):
    os.environ["BGE_API_KEY_EMBEDDINGS"] = getpass("Enter your RCP API key for embedding model (bge): ")

embeddings = OpenAIEmbeddings(
    model="BAAI/bge-m3",
    base_url="https://inference.rcp.epfl.ch/v1",
    api_key=os.getenv("BGE_API_KEY_EMBEDDINGS")
)
sparse_fr = FastEmbedSparse(model_name="Qdrant/bm25", avg_len=AVG_LEN_SPARSE_FR, language="french")

results_dir = Path(f"./dev_dataset_results_top_{K}")
results_dir.mkdir(parents=True, exist_ok=True)

In [3]:
retrieval_modes = [RetrievalMode.HYBRID, RetrievalMode.SPARSE, RetrievalMode.DENSE]

for retrieval_mode in retrieval_modes:
    qdrant = QdrantVectorStore.from_existing_collection(
        url="http://localhost:6333",
        embedding=embeddings,
        sparse_embedding=sparse_fr,
        collection_name=COLLECTION_NAME,
        retrieval_mode=retrieval_mode,
        vector_name="dense",
        sparse_vector_name="sparse_fr"
    )
    dataset = pd.read_csv('questions_dataset.csv', header=0)
    retrieval_mode_name = retrieval_mode._name_

    # save config
    config_path = results_dir / f"{retrieval_mode_name}_config.json"
    config = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "collection_name": COLLECTION_NAME,
        "retrieval_mode": retrieval_mode,
        "params_retrieval": {"k": K, "exact": True}
    }
    with open(config_path, "w", encoding="utf-8") as config_file:
        json.dump(config, config_file, ensure_ascii=False, indent=2)

    # save results
    results_path = results_dir / f"{retrieval_mode_name}_dev_results.jsonl"
    with (open(results_path, "w", encoding="utf-8") as results_file):
        for _, row in dataset.iterrows():
            if row["Dataset"] == "dev":
                query = row["Question"]
                hits = qdrant.similarity_search_with_score(query, k=K, search_params=SearchParams(exact=True))
                top_chunks = []
                for rank, (chunk, score) in enumerate(hits, start=1):
                    top_chunks.append(
                        {
                            "rank": rank,
                            "score": score,
                            "chunk_text": chunk.page_content,
                            "chunk_metadata": chunk.metadata
                        }
                    )
                result_record = {
                    "question": query,
                    "top_chunks": top_chunks,
                }
                results_file.write(json.dumps(result_record, ensure_ascii=False) + "\n")